# Ödev 2 — Model iyileştirme ve ROC eğrileri

Bu notebook ilk ödevin devamıdır. Aynı UCI Spambase veri setini kullanır. Önceki temel modelleri yeniden kurar; sonra yalnızca eğitim verisi üzerinde 5 katlı çapraz doğrulama ve `GridSearchCV` ile parametre araması yapar. En son dokunulmamış test kümesinde eski ve ayarlanmış modelleri karşılaştırır ve her algoritma için ROC eğrisi çizer.

**Önemli:** Test kümesi ayar seçmek için kullanılmaz. Böylece test sonucu daha adil kalır. Veri ham e-posta metni değil, e-postalardan çıkarılmış 57 sayısal özelliktir.

## 1. Paketleri kur
Colab'da hücrenin solundaki ▶ düğmesine bas.

In [ ]:
%pip -q install ucimlrepo

## 2. Kütüphaneler ve veri seti

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, classification_report)

spambase = fetch_ucirepo(id=94)
X = spambase.data.features
y = spambase.data.targets.iloc[:, 0].astype(int)
print('Örnek sayısı:', len(X), '| Özellik sayısı:', X.shape[1])
print('Etiketler (0=normal, 1=spam):')
print(y.value_counts().sort_index())

## 3. Aynı eğitim/test ayrımını oluştur
`stratify=y`, spam ve normal oranlarının iki parçaya da benzer dağılmasını sağlar.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print('Eğitim:', X_train.shape, '| Test:', X_test.shape)

## 4. Temel modeller ve aranacak ayarlar
Ölçekleme gereken modellerde `StandardScaler` pipeline'ın içindedir. Bu, çapraz doğrulama sırasında veri sızıntısını önler.

In [ ]:
baseline_models = {
    'Gaussian Naive Bayes': GaussianNB(),
    'Logistic Regression': Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=3000, random_state=42))]),
    'SVM (RBF)': Pipeline([('scaler', StandardScaler()), ('model', SVC(kernel='rbf'))]),
    'Random Forest': RandomForestClassifier(random_state=42, n_jobs=-1)
}

param_grids = {
    'Gaussian Naive Bayes': {'var_smoothing': [1e-10, 1e-9, 1e-8, 1e-7]},
    'Logistic Regression': {'model__C': [0.1, 1, 10, 100]},
    'SVM (RBF)': {'model__C': [0.1, 1, 10], 'model__gamma': ['scale', 0.01, 0.1]},
    'Random Forest': {
        'n_estimators': [200, 400], 'max_depth': [None, 20],
        'min_samples_leaf': [1, 2], 'max_features': ['sqrt']
    }
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

## 5. Modelleri eğit, çapraz doğrulamayla ayarla
Arama, eğitim verisinde spam sınıfının F1 skorunu yükseltmeye çalışır. Bu adım birkaç dakika sürebilir.

In [ ]:
tuned_models = {}
search_summaries = {}
for name, model in baseline_models.items():
    search = GridSearchCV(model, param_grids[name], scoring='f1', cv=cv, n_jobs=-1, refit=True)
    search.fit(X_train, y_train)
    tuned_models[name] = search.best_estimator_
    search_summaries[name] = {'En iyi CV F1': search.best_score_, 'Seçilen ayarlar': search.best_params_}
    print(f'{name}: CV F1={search.best_score_:.4f}; ayarlar={search.best_params_}')
pd.DataFrame(search_summaries).T

## 6. Dokunulmamış test kümesinde karşılaştır
Accuracy tek başına yeterli değildir; spam precision/recall/F1 ve ROC-AUC değerlerine de bakıyoruz.

In [ ]:
def score_model(model, X_part, y_part):
    pred = model.predict(X_part)
    if hasattr(model, 'predict_proba'):
        score = model.predict_proba(X_part)[:, 1]
    else:
        score = model.decision_function(X_part)
    return {
        'Accuracy': accuracy_score(y_part, pred),
        'Spam precision': precision_score(y_part, pred, zero_division=0),
        'Spam recall': recall_score(y_part, pred, zero_division=0),
        'Spam F1': f1_score(y_part, pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_part, score)
    }

rows = []
for name, model in baseline_models.items():
    model.fit(X_train, y_train)
    rows.append({'Model': name, 'Aşama': 'Temel', **score_model(model, X_test, y_test)})
for name, model in tuned_models.items():
    rows.append({'Model': name, 'Aşama': 'Ayarlanmış', **score_model(model, X_test, y_test)})
results_df = pd.DataFrame(rows)
display(results_df.sort_values(['Aşama', 'Spam F1'], ascending=[True, False]).round(4))

## 7. Ayarlanmış modellerin ROC eğrileri
ROC eğrisi, farklı karar eşiklerinde spam'i yakalama oranı (TPR) ile normal e-postalara yanlış alarm oranı (FPR) arasındaki ilişkiyi gösterir. Eğrinin sol üst köşeye yakın ve AUC'nin 1'e yakın olması daha iyi ayırt etmeye işaret eder. Kesikli çizgi rastgele tahmini gösterir.

In [ ]:
plt.figure(figsize=(9, 7))
for name, model in tuned_models.items():
    if hasattr(model, 'predict_proba'):
        scores = model.predict_proba(X_test)[:, 1]
    else:
        scores = model.decision_function(X_test)
    fpr, tpr, _ = roc_curve(y_test, scores)
    auc = roc_auc_score(y_test, scores)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Rastgele tahmin (AUC=0.5)')
plt.xlabel('Yanlış pozitif oranı (FPR)')
plt.ylabel('Doğru pozitif oranı (TPR / spam recall)')
plt.title('Ayarlanmış modellerin ROC eğrileri — test kümesi')
plt.legend(loc='lower right')
plt.grid(alpha=0.25)
plt.show()

## 8. En iyi modeli ve hata tablosunu gör
En iyi model, test kümesindeki spam F1 değerine göre seçilir. İsterseniz hocanız farklı seçim ölçütü istediyse onu kullanın. Confusion matrix'te satırlar gerçek sınıf, sütunlar tahmindir: `[[TN, FP], [FN, TP]]`.

In [ ]:
tuned_results = results_df[results_df['Aşama'] == 'Ayarlanmış'].sort_values('Spam F1', ascending=False)
best_name = tuned_results.iloc[0]['Model']
best_model = tuned_models[best_name]
best_pred = best_model.predict(X_test)
print('Test spam F1 ile en iyi ayarlanmış model:', best_name)
print(classification_report(y_test, best_pred, target_names=['Normal', 'Spam'], zero_division=0))
cm = confusion_matrix(y_test, best_pred, labels=[0, 1])
display(pd.DataFrame(cm, index=['Gerçek normal', 'Gerçek spam'], columns=['Tahmin normal', 'Tahmin spam']))
print('TN, FP, FN, TP =', cm[0,0], cm[0,1], cm[1,0], cm[1,1])
print('Not: Sonuçlar bu veri bölünmesi ve arama aralıkları için geçerlidir; başka çalıştırmalarda paket/sürüm farkları küçük oynamalar yaratabilir.')

## Rapor için kaydedilecekler
1. Temel ve ayarlanmış modellerin Accuracy, spam Precision/Recall/F1 ve ROC-AUC tablosu.
2. ROC grafiği ve AUC değerleri.
3. En yüksek test spam F1 modelinin adı ve confusion matrix'teki TN, FP, FN, TP sayıları.
4. Eğitim dışında tutulan test kümesinin model seçimi/ayar için kullanılmadığını belirtin.
5. UCI Spambase'in ham e-posta metni değil 57 sayısal özellik içerdiği sınırlamasını yazın.